In [1]:
import os
import re
import unicodedata
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import hstack, vstack
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

# =========================
# 1. Config
# =========================
CSV_PATH = "/kaggle/input/datasets/austraa/all-train/all_train_dataset_final.csv"
TEXT_COL = "bangla_speech"
LABEL_COL = "dialect"
RANDOM_STATE = 42
TRAIN_SIZE = 0.8
VALID_SIZE = 0.1
TEST_SIZE = 0.1
DROP_AMBIGUOUS_TEXTS = True

# =========================
# 2. Cleaning
# =========================
def normalize_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    text = text.replace("৷", "।")
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

def normalize_label(label: str) -> str:
    if pd.isna(label):
        return ""
    label = str(label)
    return re.sub(r"\s+", " ", label).strip()

# =========================
# 3. Load data
# =========================
def load_dataframe(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if TEXT_COL not in df.columns or LABEL_COL not in df.columns:
        raise ValueError(f"Missing columns. Found: {list(df.columns)}")

    df[TEXT_COL] = df[TEXT_COL].apply(normalize_text)
    df[LABEL_COL] = df[LABEL_COL].apply(normalize_label)
    df = df[(df[TEXT_COL].str.len() > 0) & (df[LABEL_COL].str.len() > 0)].copy()

    before = len(df)
    df = df.drop_duplicates(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
    print(f"Removed duplicate text-label rows: {before - len(df)}")

    ambiguous = df.groupby(TEXT_COL)[LABEL_COL].nunique()
    ambiguous = ambiguous[ambiguous > 1]
    print(f"Texts with multiple labels: {len(ambiguous)}")

    if DROP_AMBIGUOUS_TEXTS and len(ambiguous) > 0:
        df = df[~df[TEXT_COL].isin(set(ambiguous.index))].reset_index(drop=True)
        print(f"Dropped ambiguous texts. Rows left: {len(df)}")

    print(df[LABEL_COL].value_counts())
    return df

# =========================
# 4. Feature builder
# =========================
def build_features(train_texts, valid_texts, test_texts):
    word_vec = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        dtype=np.float32,
        max_features=250_000,
    )

    char_vec = TfidfVectorizer(
        analyzer="char",
        ngram_range=(2, 7),
        min_df=2,
        max_df=0.995,
        sublinear_tf=True,
        dtype=np.float32,
        max_features=450_000,
    )

    charwb_vec = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 7),
        min_df=2,
        max_df=0.995,
        sublinear_tf=True,
        dtype=np.float32,
        max_features=300_000,
    )

    X_train = hstack([
        word_vec.fit_transform(train_texts),
        char_vec.fit_transform(train_texts),
        charwb_vec.fit_transform(train_texts)
    ], format="csr")

    X_valid = hstack([
        word_vec.transform(valid_texts),
        char_vec.transform(valid_texts),
        charwb_vec.transform(valid_texts)
    ], format="csr")

    X_test = hstack([
        word_vec.transform(test_texts),
        char_vec.transform(test_texts),
        charwb_vec.transform(test_texts)
    ], format="csr")

    X_train = normalize(X_train, norm="l2", copy=False)
    X_valid = normalize(X_valid, norm="l2", copy=False)
    X_test = normalize(X_test, norm="l2", copy=False)

    print("Feature shapes:")
    print("X_train:", X_train.shape)
    print("X_valid:", X_valid.shape)
    print("X_test :", X_test.shape)

    return X_train, X_valid, X_test, {
        "word": word_vec,
        "char": char_vec,
        "charwb": charwb_vec,
    }

def weighted_soft_vote(prob_list, weights):
    total = np.zeros_like(prob_list[0], dtype=np.float64)
    wsum = 0.0
    for p, w in zip(prob_list, weights):
        total += p * float(w)
        wsum += float(w)
    total /= max(wsum, 1e-8)
    return total.argmax(axis=1)

# =========================
# 5. Train
# =========================
df = load_dataframe(CSV_PATH)

train_df, temp_df = train_test_split(
    df,
    test_size=(1.0 - TRAIN_SIZE),
    stratify=df[LABEL_COL],
    random_state=RANDOM_STATE,
)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=(TEST_SIZE / (VALID_SIZE + TEST_SIZE)),
    stratify=temp_df[LABEL_COL],
    random_state=RANDOM_STATE,
)

train_texts = train_df[TEXT_COL].tolist()
valid_texts = valid_df[TEXT_COL].tolist()
test_texts = test_df[TEXT_COL].tolist()

le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_valid = le.transform(valid_df[LABEL_COL])
y_test = le.transform(test_df[LABEL_COL])

X_train, X_valid, X_test, vectorizers = build_features(train_texts, valid_texts, test_texts)

svc = LinearSVC(C=3.0, class_weight="balanced", random_state=RANDOM_STATE)
svc_cal = CalibratedClassifierCV(svc, method="sigmoid", cv=3)

logreg = LogisticRegression(
    C=4.0,
    solver="saga",
    max_iter=6000,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

models = {
    "LinearSVC_calibrated": svc_cal,
    "LogReg": logreg,
}

valid_probs = []
weights = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_valid)
    pred = prob.argmax(axis=1)
    acc = accuracy_score(y_valid, pred)
    f1 = f1_score(y_valid, pred, average="macro")
    print(f"{name} -> valid_acc={acc:.4f} | valid_macro_f1={f1:.4f}")
    valid_probs.append(prob)
    weights.append(f1)
    trained_models[name] = model

ensemble_valid_pred = weighted_soft_vote(valid_probs, weights)
ensemble_valid_acc = accuracy_score(y_valid, ensemble_valid_pred)
ensemble_valid_f1 = f1_score(y_valid, ensemble_valid_pred, average="macro")
print(f"Ensemble -> valid_acc={ensemble_valid_acc:.4f} | valid_macro_f1={ensemble_valid_f1:.4f}")

X_trainvalid = vstack([X_train, X_valid], format="csr")
y_trainvalid = np.concatenate([y_train, y_valid])

final_models = {}
for name, model in models.items():
    model.fit(X_trainvalid, y_trainvalid)
    final_models[name] = model

final_probs = [m.predict_proba(X_test) for m in final_models.values()]
final_test_pred = weighted_soft_vote(final_probs, weights)

print("\n===== FINAL TEST RESULTS =====")
print("Test accuracy:", round(accuracy_score(y_test, final_test_pred), 4))
print("Test macro-F1:", round(f1_score(y_test, final_test_pred, average="macro"), 4))
print(classification_report(y_test, final_test_pred, target_names=le.classes_))

artifact = {
    "text_col": TEXT_COL,
    "label_col": LABEL_COL,
    "label_encoder": le,
    "vectorizers": vectorizers,
    "models": final_models,
    "weights": weights,
}
joblib.dump(artifact, "improved_dialect_tfidf_artifact.joblib")
print("Saved: improved_dialect_tfidf_artifact.joblib")

Removed duplicate text-label rows: 311
Texts with multiple labels: 835
Dropped ambiguous texts. Rows left: 50687
dialect
Chittagong         12784
Standard Bangla    12440
Noakhali            7939
Barisal             7121
Sylhet              5632
Mymensingh          4771
Name: count, dtype: int64
Feature shapes:
X_train: (40549, 686788)
X_valid: (5069, 686788)
X_test : (5069, 686788)
LinearSVC_calibrated -> valid_acc=0.8724 | valid_macro_f1=0.8530
LogReg -> valid_acc=0.8678 | valid_macro_f1=0.8481
Ensemble -> valid_acc=0.8710 | valid_macro_f1=0.8507

===== FINAL TEST RESULTS =====
Test accuracy: 0.8722
Test macro-F1: 0.8535
                 precision    recall  f1-score   support

        Barisal       0.86      0.86      0.86       712
     Chittagong       0.96      0.94      0.95      1279
     Mymensingh       0.74      0.74      0.74       477
       Noakhali       0.82      0.81      0.81       794
Standard Bangla       0.87      0.91      0.89      1244
         Sylhet       0.89